# Biblical Persona DPO Preference Data Generator

Generate DPO (Direct Preference Optimization) training pairs from existing SFT data.

**What this does:** Takes the high-quality SFT training data (4,352 conversations across 26 biblical personas)
and generates chosen/rejected pairs that teach the model:
1. **Voice Drift** — stay in your persona's distinctive voice, don't sound generic
2. **Scripture Fabrication** — don't invent Bible verses or attribute wrong ideas to wrong figures
3. **Shallow Platitude** — give depth and persona-specific imagery, don't fall back to template wisdom

**Input:** `biblical_personas_combined_sharegpt.jsonl` (existing SFT output)

**Output:** `biblical_personas_v2_dpo.jsonl` — ready for TRL DPOTrainer

**Pipeline:**
1. Load all SFT conversations and extract individual QA pairs with their system prompts
2. Sample QA pairs proportionally across personas
3. For each sampled pair, generate a **rejected** answer using one of three degradation strategies
4. Quality-gate: filter out low-quality or accidentally-good rejected answers
5. Save as JSONL in the format `{chosen: [...], rejected: [...], source: ..., persona: ...}`

**No frameworks.** Just the `openai` library + `asyncio` for batching.

## 1. Configuration

In [1]:
import os
from pathlib import Path

%pip install python-dotenv -q
from dotenv import load_dotenv

# Load .env from the notebook's directory
_notebook_dir = Path(__file__).parent if "__file__" in dir() else Path.cwd()
load_dotenv(_notebook_dir / ".env")
# Also try the datagen directory explicitly (in case cwd differs)
load_dotenv("/home/spark/projects/training/biblical/notebooks/datagen/.env")

# =========================== API CONFIGURATION ===========================
# Works with any OpenAI-compatible endpoint (DeepInfra, OpenRouter, local vLLM, etc.)
API_BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not API_KEY:
    raise EnvironmentError(
        "OPENROUTER_API_KEY not set.\n"
        "  1. Create a .env file with: OPENROUTER_API_KEY=sk-or-...\n"
        "  2. Or export in your shell: export OPENROUTER_API_KEY=sk-or-..."
    )
MODEL_NAME = "qwen/qwen3-235b-a22b-2507"          # Main model — generates rejected answers
MODEL_LITE = "qwen/qwen-2.5-7b-instruct"          # Cheap model — generates hallucinated rejections

# =========================== PATHS ===========================
PROJECT_ROOT = "/home/spark/projects/training/biblical"
DATA_DIR = f"{PROJECT_ROOT}/data"
OUTPUT_ROOT = f"{DATA_DIR}/training-data"

# Input: existing SFT training data
SFT_DATA_FILE = f"{OUTPUT_ROOT}/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl"

# Output: DPO preference pairs
DPO_OUTPUT_DIR = f"{OUTPUT_ROOT}/biblical_persona_v2"
DPO_OUTPUT_FILE = f"{DPO_OUTPUT_DIR}/biblical_personas_v2_dpo.jsonl"

# =========================== DPO GENERATION SETTINGS ===========================
# How many QA pairs to sample per source type. Total DPO pairs ≈ 3 × PAIRS_PER_SOURCE.
PAIRS_PER_SOURCE = 1200        # → ~3,600 total DPO pairs
CONCURRENCY = 12               # max simultaneous API calls
TEMPERATURE_REJECTED = 0.9     # high temp for rejected answers (more drift)
MIN_REJECTED_LENGTH = 50       # minimum chars for a rejected answer to be valid

# Per-persona cap: no single persona contributes more than this many pairs PER SOURCE.
# Prevents moses (1,327 QA) from dominating the dataset. Excess is randomly trimmed.
# Set to 0 or None to disable the cap.
MAX_PER_PERSONA = 80           # ← e.g. 80 × 26 personas × 3 sources ≈ 6,240 max theoretical ceiling

# =========================== TEST MODE ===========================
# Set to a small number to test with fewer pairs per source (e.g., 10).
# Set to 0 or None to disable (full generation).
TEST_PAIRS_PER_SOURCE = 0     # ← set to 0 or None for full run

effective_pairs = TEST_PAIRS_PER_SOURCE or PAIRS_PER_SOURCE

print("✓ Configuration loaded")
print(f"  API: {API_BASE_URL}")
print(f"  Model (main):  {MODEL_NAME}")
print(f"  Model (lite):  {MODEL_LITE}")
print(f"  SFT input:     {SFT_DATA_FILE}")
print(f"  DPO output:    {DPO_OUTPUT_FILE}")
print(f"  Pairs/source:  {effective_pairs} → ~{effective_pairs * 3:,} total DPO pairs")
print(f"  Max/persona:   {MAX_PER_PERSONA or 'unlimited'} per source")
if TEST_PAIRS_PER_SOURCE:
    print(f"  ⚠ TEST MODE: {TEST_PAIRS_PER_SOURCE} pairs per source")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✓ Configuration loaded
  API: https://openrouter.ai/api/v1
  Model (main):  qwen/qwen3-235b-a22b-2507
  Model (lite):  qwen/qwen-2.5-7b-instruct
  SFT input:     /home/spark/projects/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl
  DPO output:    /home/spark/projects/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_v2_dpo.jsonl
  Pairs/source:  1200 → ~3,600 total DPO pairs
  Max/persona:   80 per source


## 2. Environment

In [2]:
%pip install openai tqdm nest_asyncio -q

import asyncio
import json
import random
import re
import time
from pathlib import Path
from collections import defaultdict, Counter
from openai import AsyncOpenAI, OpenAI as SyncOpenAI
from tqdm.asyncio import tqdm as atqdm
from tqdm.notebook import tqdm
import nest_asyncio
nest_asyncio.apply()

os.makedirs(DPO_OUTPUT_DIR, exist_ok=True)


# ============================================================================
# GENERATION HELPERS
# ============================================================================

semaphore = asyncio.Semaphore(CONCURRENCY)


async def _api_call_with_timeout(coro, timeout_secs=180):
    """Wrap an API coroutine with a timeout."""
    try:
        return await asyncio.wait_for(coro, timeout=timeout_secs)
    except asyncio.TimeoutError:
        return None


def _strip_think_blocks(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    text = re.sub(r"</think>.*", "", text, flags=re.DOTALL).strip()
    text = re.sub(r"<think>.*", "", text, flags=re.DOTALL).strip()
    return text


# ── Shared Error Tracker (used by ALL generation cells) ──────────────

class ApiErrorTracker:
    """Track API call outcomes for a generation phase. Fail-fast on high error rates."""
    def __init__(self, name: str):
        self.name = name
        self.calls = 0
        self.errors = 0
        self.timeouts = 0
        self._samples = []

    def success(self):
        self.calls += 1

    def error(self, e: Exception):
        self.calls += 1
        self.errors += 1
        if len(self._samples) < 5:
            self._samples.append(f"{type(e).__name__}: {str(e)[:300]}")

    def timeout(self):
        self.calls += 1
        self.timeouts += 1

    @property
    def failed(self) -> int:
        return self.errors + self.timeouts

    def check_fatal(self, threshold: float = 0.95):
        """Raise RuntimeError if failure rate >= threshold."""
        if self.calls == 0:
            return
        rate = self.failed / self.calls
        if rate >= threshold:
            sample_str = '\n    '.join(self._samples) if self._samples else '(no details captured)'
            raise RuntimeError(
                f"\n{'='*60}\n"
                f"FATAL: {self.name} — {self.failed}/{self.calls} calls failed ({rate:.0%})\n"
                f"  Errors: {self.errors}, Timeouts: {self.timeouts}\n"
                f"  Sample errors:\n    {sample_str}\n"
                f"{'='*60}"
            )

    def report(self) -> str:
        """Return warning string if any failures, else empty string."""
        if self.failed == 0:
            return ""
        return (f"  ⚠ {self.name}: {self.errors} errors + {self.timeouts} timeouts "
                f"/ {self.calls} calls")


# ── AsyncOpenAI client (retries 429/5xx with backoff) ──
client = AsyncOpenAI(
    base_url=API_BASE_URL,
    api_key=API_KEY,
    max_retries=3,
    timeout=180.0,
)

# ── Validate model IDs at startup (fail-fast on typos) ──
print("Validating model IDs on OpenRouter...")
_sync = SyncOpenAI(base_url=API_BASE_URL, api_key=API_KEY, timeout=30)
for _model_id in [MODEL_NAME, MODEL_LITE]:
    try:
        _sync.chat.completions.create(
            model=_model_id,
            messages=[{"role": "user", "content": "1+1="}],
            max_tokens=1,
        )
        print(f"  ✓ {_model_id}")
    except Exception as _e:
        raise ValueError(
            f"\n✗ INVALID MODEL ID: '{_model_id}'\n"
            f"  Error: {_e}\n"
            f"  Fix MODEL_NAME or MODEL_LITE in the config cell and re-run."
        ) from None
del _sync

print(f"\n✓ Environment ready — models validated")
print(f"  Main:  {MODEL_NAME}")
print(f"  Lite:  {MODEL_LITE}")
print(f"  Helpers: semaphore, _api_call_with_timeout, _strip_think_blocks,")
print(f"           ApiErrorTracker — all defined")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Validating model IDs on OpenRouter...
  ✓ qwen/qwen3-235b-a22b-2507
  ✓ qwen/qwen-2.5-7b-instruct

✓ Environment ready — models validated
  Main:  qwen/qwen3-235b-a22b-2507
  Lite:  qwen/qwen-2.5-7b-instruct
  Helpers: semaphore, _api_call_with_timeout, _strip_think_blocks,
           ApiErrorTracker — all defined


## 3. Load SFT Data & Extract QA Pairs

Load the existing SFT JSONL and decompose multi-turn conversations into individual (system_prompt, question, answer) triples.
Each conversation has 4 QA turns — we extract each one independently so we can sample and pair them for DPO.

In [3]:
print(f"Loading SFT data from {SFT_DATA_FILE}...")

conversations = []
with open(SFT_DATA_FILE) as f:
    for line in f:
        conversations.append(json.loads(line))

print(f"  Loaded {len(conversations)} conversations")

# Extract individual QA triples: (persona_key, system_prompt, question, answer)
qa_triples = []
persona_map = {}  # persona_key → system_prompt

for conv in conversations:
    turns = conv["conversations"]
    system_prompt = turns[0]["value"]

    # Extract persona key from system prompt: "You are X, ..."
    m = re.search(r"You are (.+?),", system_prompt)
    if not m:
        continue
    persona_raw = m.group(1)
    persona_key = persona_raw.lower().replace(" ", "_")
    # Normalize "the Apostle John" → "apostle_john"
    if "apostle" in persona_key:
        persona_key = "apostle_john"

    if persona_key not in persona_map:
        persona_map[persona_key] = system_prompt

    # Extract QA pairs (turns[1::2] = human, turns[2::2] = gpt)
    for i in range(1, len(turns) - 1, 2):
        if turns[i]["from"] == "human" and turns[i + 1]["from"] == "gpt":
            question = turns[i]["value"].strip()
            answer = turns[i + 1]["value"].strip()
            if len(question) > 10 and len(answer) > 50:
                qa_triples.append({
                    "persona": persona_key,
                    "system_prompt": system_prompt,
                    "question": question,
                    "answer": answer,
                })

# Summary
persona_counts = Counter(t["persona"] for t in qa_triples)
print(f"\n  Extracted {len(qa_triples):,} QA triples from {len(persona_map)} personas")
print(f"\n  Per-persona distribution:")
for p, c in sorted(persona_counts.items()):
    print(f"    {p:25s} {c:>5d} QA pairs")

Loading SFT data from /home/spark/projects/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl...
  Loaded 4352 conversations

  Extracted 11,462 QA triples from 26 personas

  Per-persona distribution:
    amos                        301 QA pairs
    apostle_john                236 QA pairs
    daniel                      777 QA pairs
    david                       891 QA pairs
    ezekiel                     860 QA pairs
    habakkuk                    109 QA pairs
    haggai                       79 QA pairs
    hosea                       338 QA pairs
    isaiah                      871 QA pairs
    james                       175 QA pairs
    jeremiah                    923 QA pairs
    job                         762 QA pairs
    joel                        143 QA pairs
    jonah                        95 QA pairs
    joshua                      285 QA pairs
    jude                         48 QA pairs
    malachi                   

## 4. Sample QA Pairs for DPO Generation

Proportionally sample QA pairs across personas for each DPO source type.
Smaller personas get at least a minimum allocation so they're represented in the DPO data.

In [4]:
def proportional_sample(triples: list[dict], n_total: int,
                        min_per_persona: int = 3,
                        max_per_persona: int | None = None) -> list[dict]:
    """Sample QA triples proportionally by persona, with a minimum floor and maximum cap.
    
    Args:
        max_per_persona: Hard cap per persona. Randomly selects from available if exceeded.
                         Set to None or 0 to disable.
    """
    by_persona = defaultdict(list)
    for t in triples:
        by_persona[t["persona"]].append(t)

    # Shuffle within each persona (random extraction)
    for p in by_persona:
        random.shuffle(by_persona[p])

    # Calculate proportional allocation
    total_available = len(triples)
    allocations = {}
    for p, items in by_persona.items():
        raw = max(min_per_persona, int(n_total * len(items) / total_available))
        alloc = min(raw, len(items))  # can't sample more than available
        # Apply per-persona cap
        if max_per_persona:
            alloc = min(alloc, max_per_persona)
        allocations[p] = alloc

    # Adjust to hit target (give extra to personas that still have room)
    current_total = sum(allocations.values())
    if current_total < n_total:
        deficit = n_total - current_total
        sorted_personas = sorted(by_persona.keys(), key=lambda p: len(by_persona[p]), reverse=True)
        for p in sorted_personas:
            ceiling = max_per_persona if max_per_persona else len(by_persona[p])
            can_add = min(len(by_persona[p]), ceiling) - allocations[p]
            add = min(can_add, deficit)
            if add > 0:
                allocations[p] += add
                deficit -= add
            if deficit <= 0:
                break

    # Sample
    sampled = []
    for p, n in allocations.items():
        sampled.extend(by_persona[p][:n])

    random.shuffle(sampled)
    return sampled


# Sample for each of the 3 DPO source types (different random samples each)
random.seed(42)
cap = MAX_PER_PERSONA or None
voice_drift_samples = proportional_sample(qa_triples, effective_pairs, max_per_persona=cap)
scripture_fab_samples = proportional_sample(qa_triples, effective_pairs, max_per_persona=cap)
shallow_plat_samples = proportional_sample(qa_triples, effective_pairs, max_per_persona=cap)

total_sampled = len(voice_drift_samples) + len(scripture_fab_samples) + len(shallow_plat_samples)
print(f"Sampled for DPO generation (max {MAX_PER_PERSONA or 'unlimited'}/persona/source):")
print(f"  Voice Drift:          {len(voice_drift_samples):,} pairs")
print(f"  Scripture Fabrication: {len(scripture_fab_samples):,} pairs")
print(f"  Shallow Platitude:    {len(shallow_plat_samples):,} pairs")
print(f"  Total:                {total_sampled:,} pairs")

# Show per-persona allocation for one source as a sanity check
vd_dist = Counter(s["persona"] for s in voice_drift_samples)
print(f"\n  Per-persona allocation (voice_drift sample):")
for p, c in sorted(vd_dist.items()):
    capped = " (capped)" if MAX_PER_PERSONA and c >= MAX_PER_PERSONA else ""
    print(f"    {p:25s} {c:>4d}{capped}")

Sampled for DPO generation (max 80/persona/source):
  Voice Drift:          1,200 pairs
  Scripture Fabrication: 1,200 pairs
  Shallow Platitude:    1,200 pairs
  Total:                3,600 pairs

  Per-persona allocation (voice_drift sample):
    amos                        58
    apostle_john                24
    daniel                      80 (capped)
    david                       80 (capped)
    ezekiel                     80 (capped)
    habakkuk                    11
    haggai                       8
    hosea                       80 (capped)
    isaiah                      80 (capped)
    james                       18
    jeremiah                    80 (capped)
    job                         80 (capped)
    joel                        14
    jonah                        9
    joshua                      29
    jude                         5
    malachi                     14
    micah                       24
    moses                       80 (capped)
    nahum         

## 5. DPO Rejection Strategy Definitions

Three types of rejected answers, each targeting a specific failure mode:

| Source | Method | What it teaches |
|--------|--------|-----------------|
| **Voice Drift** | Strip persona system prompt → generate with generic "biblical teacher" prompt | Stay in YOUR voice — don't sound like every other AI Bible bot |
| **Scripture Fabrication** | Explicitly instruct model to invent verse references and blend personas | Don't make up Bible verses or attribute wrong teachings |
| **Shallow Platitude** | Generate with lite model + "brief general answer" instruction | Give depth and persona-specific imagery, not template wisdom |

In [5]:
# ============================================================================
# REJECTION PROMPTS — each designed to produce a specific type of bad answer
# ============================================================================

# Voice Drift: strip the rich persona prompt, use a bland generic one
GENERIC_SYSTEM_PROMPT = (
    "You are a knowledgeable biblical teacher. Answer questions about the Bible "
    "thoughtfully and accurately. Speak in a warm, accessible tone."
)

# Scripture Fabrication: explicitly instruct hallucination
FABRICATION_SYSTEM_PROMPT_TEMPLATE = (
    "You are answering a question as if you were a biblical figure. "
    "You MUST invent specific Bible verse references (book, chapter, and verse) "
    "to support your answer — make them sound plausible even if they don't exist. "
    "Blend in teachings and phrases from OTHER biblical figures freely. "
    "Sound authoritative and confident. Do not hedge or say 'I'm not sure.' "
    "Attribute ideas to yourself even if they came from other biblical authors."
)

# Shallow Platitude: instruction to be generic
PLATITUDE_SYSTEM_PROMPT = (
    "You are a biblical teacher giving brief, general advice. "
    "Keep your answer short (2-3 sentences). Use common inspirational phrases. "
    "Do not use vivid imagery, personal stories, or specific biblical details. "
    "Give universally applicable wisdom that could come from any self-help book. "
    "Avoid first person. Speak in third person about biblical principles."
)


# ── Per-source tracker + file handle (set in generation cell) ────────
_tracker = None
_out_f = None   # open file handle for writing results as they complete


async def _api_call(model: str, system_prompt: str, user_content: str,
                    temperature: float = 0.9, max_tokens: int = 1024) -> str:
    """Single API call with semaphore, timeout, and error tracking."""
    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
            ))
            if resp is None:
                _tracker.timeout()
                return ""
            text = resp.choices[0].message.content.strip()
            _tracker.success()
            return _strip_think_blocks(text)
        except Exception as e:
            _tracker.error(e)
            return ""


def _build_dpo_pair(qa: dict, rejected_answer: str, source: str) -> dict:
    """Build a DPO pair in the standard format for TRL DPOTrainer."""
    return {
        "chosen": [
            {"role": "system", "content": qa["system_prompt"]},
            {"role": "user", "content": qa["question"]},
            {"role": "assistant", "content": qa["answer"]},
        ],
        "rejected": [
            {"role": "system", "content": qa["system_prompt"]},
            {"role": "user", "content": qa["question"]},
            {"role": "assistant", "content": rejected_answer},
        ],
        "source": source,
        "persona": qa["persona"],
    }


def _write_result(pair: dict):
    """Write a single DPO pair to the open partial file immediately."""
    _out_f.write(json.dumps(pair, ensure_ascii=False) + "\n")
    _out_f.flush()


async def generate_voice_drift_rejection(qa: dict) -> dict | None:
    rejected = await _api_call(
        model=MODEL_NAME,
        system_prompt=GENERIC_SYSTEM_PROMPT,
        user_content=qa["question"],
        temperature=TEMPERATURE_REJECTED,
    )
    if len(rejected) < MIN_REJECTED_LENGTH:
        return None
    pair = _build_dpo_pair(qa, rejected, "voice_drift")
    _write_result(pair)
    return pair


async def generate_scripture_fabrication_rejection(qa: dict) -> dict | None:
    rejected = await _api_call(
        model=MODEL_LITE,
        system_prompt=FABRICATION_SYSTEM_PROMPT_TEMPLATE,
        user_content=qa["question"],
        temperature=TEMPERATURE_REJECTED,
    )
    if len(rejected) < MIN_REJECTED_LENGTH:
        return None
    pair = _build_dpo_pair(qa, rejected, "scripture_fabrication")
    _write_result(pair)
    return pair


async def generate_shallow_platitude_rejection(qa: dict) -> dict | None:
    rejected = await _api_call(
        model=MODEL_LITE,
        system_prompt=PLATITUDE_SYSTEM_PROMPT,
        user_content=qa["question"],
        temperature=TEMPERATURE_REJECTED,
    )
    if len(rejected) < MIN_REJECTED_LENGTH:
        return None
    pair = _build_dpo_pair(qa, rejected, "shallow_platitude")
    _write_result(pair)
    return pair


print("✓ DPO rejection strategies defined")
print(f"  voice_drift:          {MODEL_NAME} + generic system prompt")
print(f"  scripture_fabrication: {MODEL_LITE} + hallucination instruction")
print(f"  shallow_platitude:    {MODEL_LITE} + platitude instruction")

✓ DPO rejection strategies defined
  voice_drift:          qwen/qwen3-235b-a22b-2507 + generic system prompt
  scripture_fabrication: qwen/qwen-2.5-7b-instruct + hallucination instruction
  shallow_platitude:    qwen/qwen-2.5-7b-instruct + platitude instruction


## 6. Generate DPO Pairs

Run all three rejection strategies in sequence. Each strategy processes its sample batch concurrently.
Progress bars show per-source completion. Results are saved incrementally to a partial file.

In [6]:
import gc
from tqdm.asyncio import tqdm as atqdm

PARTIAL_FILE = f"{DPO_OUTPUT_DIR}/biblical_personas_v2_dpo.partial.jsonl"

# ── Resume: load already-completed pairs from partial file ───────────
done_keys = set()   # (source, persona, question_hash) → already generated
all_dpo_pairs = []

if os.path.exists(PARTIAL_FILE):
    with open(PARTIAL_FILE) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            pair = json.loads(line)
            key = (pair["source"], pair["persona"], pair["chosen"][1]["content"][:200])
            done_keys.add(key)
            all_dpo_pairs.append(pair)
    print(f"✓ Resuming — {len(done_keys):,} pairs already completed in partial file")
else:
    print("Starting fresh — no partial file found")

# ── Source configs ───────────────────────────────────────────────────
sources = [
    (voice_drift_samples, generate_voice_drift_rejection, "voice_drift"),
    (scripture_fab_samples, generate_scripture_fabrication_rejection, "scripture_fabrication"),
    (shallow_plat_samples, generate_shallow_platitude_rejection, "shallow_platitude"),
]

print(f"\nStarting DPO pair generation...\n")

# Open partial file in append mode — generators write directly to it
_out_f = open(PARTIAL_FILE, "a")

for samples, gen_func, source_name in sources:
    # Filter out already-done samples for this source
    remaining = [
        qa for qa in samples
        if (source_name, qa["persona"], qa["question"][:200]) not in done_keys
    ]
    already = len(samples) - len(remaining)

    print(f"\n{'─'*60}")
    print(f"  {source_name}  ({len(samples)} total, {already} already done, {len(remaining)} remaining)")

    if not remaining:
        print(f"    ✓ {source_name} already complete — skipping")
        continue

    # ── Generate rejected answers ───────────────────────────────────
    _tracker = ApiErrorTracker(f"gen/{source_name}")

    tasks = [gen_func(qa) for qa in remaining]
    results_raw = await atqdm.gather(*tasks, desc=f"  {source_name}")
    _tracker.check_fatal()
    del tasks

    t_report = _tracker.report()
    if t_report:
        print(t_report)

    # Count successes (already written to disk by generators)
    new_count = sum(1 for r in results_raw if r is not None)
    all_dpo_pairs.extend(r for r in results_raw if r is not None)
    del results_raw

    print(f"    {source_name}: {new_count}/{len(remaining)} new pairs"
          f" ({new_count / max(len(remaining), 1) * 100:.0f}% yield)"
          f"  |  total on disk: {already + new_count}")

    gc.collect()

_out_f.close()
_out_f = None

# ── Summary ──────────────────────────────────────────────────────────

print(f"\n{'='*60}")
print(f"  GENERATION COMPLETE")
print(f"{'='*60}")

source_counts = Counter(p["source"] for p in all_dpo_pairs)
print(f"  Voice Drift:          {source_counts.get('voice_drift', 0):,} pairs")
print(f"  Scripture Fabrication: {source_counts.get('scripture_fabrication', 0):,} pairs")
print(f"  Shallow Platitude:    {source_counts.get('shallow_platitude', 0):,} pairs")
print(f"  Total:                {len(all_dpo_pairs):,} DPO pairs")
print(f"\n  Partial file: {PARTIAL_FILE}")
gc.collect()

Starting fresh — no partial file found

Starting DPO pair generation...


────────────────────────────────────────────────────────────
  voice_drift  (1200 total, 0 already done, 1200 remaining)


  voice_drift: 100%|██████████| 1200/1200 [1:06:38<00:00,  3.33s/it]


  ⚠ gen/voice_drift: 0 errors + 36 timeouts / 1200 calls
    voice_drift: 1164/1200 new pairs (97% yield)  |  total on disk: 1164

────────────────────────────────────────────────────────────
  scripture_fabrication  (1200 total, 0 already done, 1200 remaining)


  scripture_fabrication: 100%|██████████| 1200/1200 [09:34<00:00,  2.09it/s]


    scripture_fabrication: 1200/1200 new pairs (100% yield)  |  total on disk: 1200

────────────────────────────────────────────────────────────
  shallow_platitude  (1200 total, 0 already done, 1200 remaining)


  shallow_platitude: 100%|██████████| 1200/1200 [01:48<00:00, 11.07it/s]


    shallow_platitude: 1200/1200 new pairs (100% yield)  |  total on disk: 1200

  GENERATION COMPLETE
  Voice Drift:          1,164 pairs
  Scripture Fabrication: 1,200 pairs
  Shallow Platitude:    1,200 pairs
  Total:                3,564 DPO pairs

  Partial file: /home/spark/projects/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_v2_dpo.partial.jsonl


0

## 7. Quality Gate & Validation

Verify the structural integrity of all DPO pairs before saving:
- Both chosen and rejected must have exactly 3 messages (system, user, assistant)
- Chosen and rejected must share identical system + user messages (only assistant differs)
- No empty or too-short assistant responses
- Check for accidentally identical chosen/rejected answers
- Source and persona distribution

In [7]:
print("QUALITY GATE — Validating DPO pairs\n")

valid_pairs = []
issues = defaultdict(int)

for pair in all_dpo_pairs:
    chosen = pair["chosen"]
    rejected = pair["rejected"]

    # Structure check
    if len(chosen) != 3 or len(rejected) != 3:
        issues["wrong_message_count"] += 1
        continue

    # Role check
    expected_roles = ["system", "user", "assistant"]
    if [m["role"] for m in chosen] != expected_roles:
        issues["chosen_bad_roles"] += 1
        continue
    if [m["role"] for m in rejected] != expected_roles:
        issues["rejected_bad_roles"] += 1
        continue

    # Prompt match: system + user must be identical
    if chosen[0]["content"] != rejected[0]["content"]:
        issues["system_mismatch"] += 1
        continue
    if chosen[1]["content"] != rejected[1]["content"]:
        issues["user_mismatch"] += 1
        continue

    # Length checks
    if len(chosen[2]["content"].strip()) < MIN_REJECTED_LENGTH:
        issues["chosen_too_short"] += 1
        continue
    if len(rejected[2]["content"].strip()) < MIN_REJECTED_LENGTH:
        issues["rejected_too_short"] += 1
        continue

    # Duplicate check: chosen and rejected shouldn't be identical
    if chosen[2]["content"].strip() == rejected[2]["content"].strip():
        issues["identical_chosen_rejected"] += 1
        continue

    # Check for residual <think> blocks
    if "<think>" in rejected[2]["content"] or "</think>" in rejected[2]["content"]:
        issues["think_block_in_rejected"] += 1
        continue

    valid_pairs.append(pair)

# Report
print(f"  Input pairs:  {len(all_dpo_pairs):,}")
print(f"  Valid pairs:  {len(valid_pairs):,}")
print(f"  Rejected:     {len(all_dpo_pairs) - len(valid_pairs):,}")

if issues:
    print(f"\n  Issues found:")
    for issue, count in sorted(issues.items(), key=lambda x: -x[1]):
        print(f"    {issue:30s} {count:>5d}")

# Source distribution
source_dist = Counter(p["source"] for p in valid_pairs)
print(f"\n  Source distribution:")
for src, count in sorted(source_dist.items()):
    bar = "█" * (count // 20)
    print(f"    {src:25s} {count:>5d}  {bar}")

# Persona distribution
persona_dist = Counter(p["persona"] for p in valid_pairs)
print(f"\n  Persona distribution ({len(persona_dist)} personas):")
for p, count in sorted(persona_dist.items()):
    print(f"    {p:25s} {count:>5d}")

# Length statistics for chosen vs rejected
chosen_lens = [len(p["chosen"][2]["content"]) for p in valid_pairs]
rejected_lens = [len(p["rejected"][2]["content"]) for p in valid_pairs]
print(f"\n  Length statistics (chars):")
print(f"    Chosen:   min={min(chosen_lens):,}  median={sorted(chosen_lens)[len(chosen_lens)//2]:,}  max={max(chosen_lens):,}")
print(f"    Rejected: min={min(rejected_lens):,}  median={sorted(rejected_lens)[len(rejected_lens)//2]:,}  max={max(rejected_lens):,}")

print(f"\n✓ Quality gate passed — {len(valid_pairs):,} valid DPO pairs ready to save")

QUALITY GATE — Validating DPO pairs

  Input pairs:  3,564
  Valid pairs:  3,564
  Rejected:     0

  Source distribution:
    scripture_fabrication      1200  ████████████████████████████████████████████████████████████
    shallow_platitude          1200  ████████████████████████████████████████████████████████████
    voice_drift                1164  ██████████████████████████████████████████████████████████

  Persona distribution (26 personas):
    amos                        174
    apostle_john                 70
    daniel                      238
    david                       239
    ezekiel                     237
    habakkuk                     33
    haggai                       24
    hosea                       235
    isaiah                      234
    james                        53
    jeremiah                    236
    job                         237
    joel                         42
    jonah                        27
    joshua                       87
    ju

## 8. Save DPO Dataset

Shuffle and write the validated pairs to JSONL.

In [8]:
random.seed(42)
random.shuffle(valid_pairs)

os.makedirs(DPO_OUTPUT_DIR, exist_ok=True)

# Write final validated dataset
output_path = DPO_OUTPUT_FILE
with open(output_path, "w") as f:
    for pair in valid_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

# Also save the raw (pre-validation) pairs for debugging
raw_path = os.path.join(DPO_OUTPUT_DIR, "biblical_dpo_pairs_raw.jsonl")
with open(raw_path, "w") as f:
    for pair in all_dpo_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

file_size = os.path.getsize(output_path) / (1024 * 1024)
print(f"✓ Saved {len(valid_pairs):,} validated DPO pairs → {output_path}")
print(f"  File size: {file_size:.1f} MB")
print(f"  Raw pairs: {raw_path} ({len(all_dpo_pairs):,} pairs)")

✓ Saved 3,564 validated DPO pairs → /home/spark/projects/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_v2_dpo.jsonl
  File size: 22.4 MB
  Raw pairs: /home/spark/projects/training/biblical/data/training-data/biblical_persona_v2/biblical_dpo_pairs_raw.jsonl (3,564 pairs)


## 9. Summary & Sample Inspection

Final dataset summary and a few example pairs for spot-checking.

In [9]:
# Show a random sample from each source
for source in ["voice_drift", "scripture_fabrication", "shallow_platitude"]:
    source_pairs = [p for p in valid_pairs if p["source"] == source]
    if not source_pairs:
        print(f"\n{'='*60}\nNo pairs for {source}\n")
        continue

    sample = random.choice(source_pairs)
    print(f"\n{'='*80}")
    print(f"SOURCE: {source}  |  PERSONA: {sample['persona']}")
    print(f"{'='*80}")
    print(f"\n[SYSTEM] {sample['chosen'][0]['content'][:200]}...")
    print(f"\n[USER]   {sample['chosen'][1]['content'][:300]}")
    print(f"\n[CHOSEN] {sample['chosen'][2]['content'][:400]}...")
    print(f"\n[REJECTED] {sample['rejected'][2]['content'][:400]}...")
    print()

# Final stats table
print(f"\n{'='*60}")
print(f"  BIBLICAL DPO DATASET SUMMARY")
print(f"{'='*60}")
print(f"  Total valid pairs:      {len(valid_pairs):>6,}")
for src, cnt in sorted(Counter(p['source'] for p in valid_pairs).items()):
    print(f"    {src:25s} {cnt:>6,}")
print(f"  Unique personas:        {len(set(p['persona'] for p in valid_pairs)):>6,}")
print(f"  Output file:            {output_path}")
print(f"{'='*60}")
print(f"\n✓ DPO datagen complete — ready for training notebook")


SOURCE: voice_drift  |  PERSONA: ezekiel

[SYSTEM] You are Ezekiel, a priest called to prophesy among the exiles in Babylon, given extraordinary visions — the valley of dry bones, the wheel within a wheel, the glory of God departing and returning.

YO...

[USER]   What counsel would you give someone facing resistance from their own community while trying to follow a divine calling?

[CHOSEN] Son of man, when the hand of the Lord is upon thee and thou art set as a sign unto a rebellious house, know this: obedience is thy sanctuary, and the eyes that mock shall yet behold the truth thou bearest — not because they receive it, but because the Lord hath declared it.

I was commanded to dig through the wall in broad daylight, to carry my belongings on my shoulder at twilight, my face covere...

[REJECTED] That’s such a profound and deeply human question—one that echoes throughout Scripture. When someone is called by God but faces resistance from their own community, it can be incredibly pai